# 06_analysis_comparison.ipynb

## Analysis & Comparison: Grid vs Financial

This notebook compares results from both experiments:

- **Experiment 1 (Grid/Stateless)**: IEEE 118-bus power flow prediction
- **Experiment 2 (Financial/Stateful)**: S&P 500 direction prediction

Key comparison: Replicability Score in stateless vs stateful environments

## 1. Load All Results

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_DIR = Path("/home/l1zle/LLMIP")
RESULTS_DIR = PROJECT_DIR / "results"

# Load all results
with open(RESULTS_DIR / "xgboost_grid_results.json") as f:
    xgb_grid = json.load(f)

with open(RESULTS_DIR / "xgboost_financial_results.json") as f:
    xgb_financial = json.load(f)

with open(RESULTS_DIR / "grid_replicability_test.json") as f:
    llmip_grid = json.load(f)

with open(RESULTS_DIR / "financial_replicability_test.json") as f:
    llmip_financial = json.load(f)

print("=== All Results Loaded ===")
print(f"Grid XGBoost: {xgb_grid['metrics']['mae_overall']:.4f} MAE")
print(f"Grid LLM: {llmip_grid['original_llm_performance']['mae_overall']:.4f} MAE")
print(f"Financial XGBoost: {xgb_financial['metrics']['accuracy']:.1%} accuracy")
print(f"Financial LLM: {llmip_financial['original_accuracy']:.1%} accuracy")
print(f"Financial Replicability Score: {llmip_financial['replicability_score']:.1%}")

## 2. Performance Comparison Table

In [ ]:
print("\n" + "="*80)
print("PERFORMANCE COMPARISON: GRID vs FINANCIAL")
print("="*80)

print("\n{:<30} {:<20} {:<20}".format("Metric", "Grid (Stateless)", "Financial (Stateful)"))
print("-"*80)
print("{:<30} {:<20} {:<20}".format("XGBoost MAE", f"{xgb_grid['metrics']['mae_overall']:.4f} pu", "N/A (classification)"))
print("{:<30} {:<20} {:<20}".format("XGBoost Accuracy", "N/A (regression)", f"{xgb_financial['metrics']['accuracy']:.1%}"))
print("{:<30} {:<20} {:<20}".format("LLM MAE", f"{llmip_grid['original_llm_performance']['mae_overall']:.4f} pu", "N/A"))
print("{:<30} {:<20} {:<20}".format("LLM Accuracy", "N/A", f"{llmip_financial['original_accuracy']:.1%}"))
print("{:<30} {:<20} {:<20}".format("Replicability Score", "Pending", f"{llmip_financial['replicability_score']:.1%}"))

## 3. Key Findings

In [ ]:
print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80)

# Grid findings
xgb_mae = xgb_grid['metrics']['mae_overall']
llm_mae = llmip_grid['original_llm_performance']['mae_overall']
ratio = llm_mae / xgb_mae if xgb_mae > 0 else 0

print("\n### GRID (Stateless) ###")
print(f"1. XGBoost MAE: {xgb_mae:.4f} pu")
print(f"2. LLM MAE: {llm_mae:.4f} pu")
print(f"3. LLM is {ratio:.0f}x worse than XGBoost on numerical accuracy")
print("4. LLM captures qualitative physics but lacks numerical precision")

# Financial findings
xgb_acc = xgb_financial['metrics']['accuracy']
llm_acc = llmip_financial['original_accuracy']
r_score = llmip_financial['replicability_score']

print("\n### FINANCIAL (Stateful) ###")
print(f"1. XGBoost accuracy: {xgb_acc:.1%} (essentially random)")
print(f"2. LLM accuracy: {llm_acc:.1%}")
print(f"3. Replicability Score: {r_score:.1%}")
print(f"4. Fresh LLM (with Rulebook) outperformed original LLM: {r_score > llm_acc}")

## 4. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Accuracy Comparison
ax1 = axes[0]
categories = ['XGBoost\nFinancial', 'LLM\nFinancial', 'Fresh LLM\n(Replicability)']
values = [xgb_acc * 100, llm_acc * 100, r_score * 100]
colors = ['#2ecc71', '#3498db', '#e74c3c']
bars = ax1.bar(categories, values, color=colors, edgecolor='black')
ax1.axhline(50, color='gray', linestyle='--', label='Random (50%)')
ax1.set_ylabel('Accuracy (%)')
ax1.set_title('Financial (Stateful) Experiment\nAccuracy Comparison')
ax1.set_ylim(0, 100)
ax1.legend()
for bar, val in zip(bars, values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{val:.0f}%', ha='center')

# Plot 2: Grid Error Comparison
ax2 = axes[1]
categories = ['XGBoost', 'LLM']
values = [xgb_mae * 100, llm_mae * 100]
colors = ['#2ecc71', '#3498db']
bars = ax2.bar(categories, values, color=colors, edgecolor='black')
ax2.set_ylabel('Mean Absolute Error (×100)')
ax2.set_title('Grid (Stateless) Experiment\nMAE Comparison')
for bar, val in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, f'{val:.2f}', ha='center')

# Plot 3: Summary
ax3 = axes[2]
summary_text = """
KEY INSIGHTS:

Grid (Stateless):
- XGBoost >> LLM on accuracy
- LLM captures physics qualitatively

Financial (Stateful):
- Both ~random (50% baseline)
- Replicability Score = 60%
- Fresh LLM benefits from rules

Conclusion:
- Stateful environments are harder
- Rulebook provides structure
- Numerical precision remains weak
"""
ax3.text(0.1, 0.5, summary_text, transform=ax3.transAxes, fontsize=10,
        verticalalignment='center', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax3.axis('off')
ax3.set_title('Summary')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'comparison_grid_vs_financial.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFigure saved to: {RESULTS_DIR / 'comparison_grid_vs_financial.png'}")

## 5. Thesis Alignment

In [ ]:
print("\n" + "="*80)
print("THESIS ALIGNMENT CHECK")
print("="*80)

print("\n### What the Thesis Claimed ###\n")
print("1. LLMIP produces competitive prediction with full explainability")
print("2. Replicability Score is a novel evaluation metric")
print("3. Stateless vs stateful comparison characterizes LLM reasoning limits")
print("4. Rulebook captures genuine predictive logic")

print("### What We Observed ###\n")
print("1. Prediction accuracy: XGBoost >> LLM in both domains")
print("2. Replicability Score: 60% in financial (above random baseline)")
print("3. Stateful degradation: Both XGBoost and LLM struggle in financial")
print("4. Rulebook: Fresh LLM can follow rules, suggesting logic capture")

print("### Paper Framing ###\n")
print("STRENGTHEN: Emphasize the Replicability Score as novel contribution")
print("WEAKEN: Do not overclaim on prediction accuracy vs XGBoost")
print("EMPHASIZE: Stateful vs stateless degradation is the key finding")

## 6. Next Steps

In [ ]:
print("\n" + "="*80)
print("RECOMMENDED NEXT STEPS")
print("="*80)

print("\n1. IMPROVE REPLICABILITY TEST")
print("   - Fix parsing to get valid Replicability Score for grid")
print("   - Run more samples for statistical significance")

print("\n2. PAPER WRITING")
print("   - Focus on Replicability Score as novel contribution")
print("   - Position as evaluation metric, not just prediction tool")
print("   - Acknowledge accuracy trade-off honestly")

print("\n3. EXTENSIONS")
print("   - Prompt ablation study (3-5 prompt variations)")
print("   - Human evaluation with domain experts")
print("   - Physical consistency check for grid rulebook")

print("\n4. CONFERENCE TARGET")
print("   - PAKDD main track seems achievable")
print("   - Focus on Replicability Score as contribution")
print("   - Stateless vs stateful comparison as key insight")